In [ ]:
import os
import pandas as pd
import psycopg2
from pathlib import Path
from dotenv import load_dotenv

# Mesma configuração de .env da rotina de agregação
env_path = r"C:\Users\zeno.filho\projetos\python_secrets_lib\.env"
load_dotenv(dotenv_path=env_path)


def get_db_connection():
    return psycopg2.connect(
        host=os.getenv("DB_HOST"),
        database=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        port=os.getenv("DB_PORT", 5432),
    )


def run_sql(sql, params=None, fetch=True):
    """Executa SQL e retorna DataFrame (fetch=True) ou nada."""
    conn = get_db_connection()
    try:
        cur = conn.cursor()
        cur.execute(sql, params)
        if fetch:
            cols = [d[0] for d in cur.description]
            rows = cur.fetchall()
            cur.close()
            return pd.DataFrame(rows, columns=cols)
        else:
            conn.commit()
            cur.close()
    except Exception:
        conn.rollback()
        raise
    finally:
        if not conn.closed:
            conn.close()


# Teste
run_sql("SELECT current_database(), current_user;")

,current_database,current_user
0,dbs_surod,zeno.andrade


## 1. Parâmetros de exportação

Edite aqui para trocar a matriz a ser exportada.

In [2]:
# Tabela de origem (matriz agregada)
MTX_SCHEMA = "VISUM"
MTX_TABELA = "mtx_od_pequi"
MTX_COL_O = "o"
MTX_COL_D = "d"
MTX_COL_VIAGENS = "viagens"

# Pasta de saída para os arquivos
OUTPUT_DIR = Path(r"C:\Users\zeno.filho\Downloads\matrizes_exportadas")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Nome base dos arquivos (sem extensão)
NOME_BASE = "mtx_od_pequi"

# Configurações do .fma
INCLUIR_ZEROS = True         # True = todos os N² pares; False = só pares com viagens > 0
CASAS_DECIMAIS = 3           # precisão dos valores de viagens no .fma
FATOR_MATRIX = 1.0           # fator multiplicador (linha "Matrix" do .fma)

# Comentário opcional que vai no cabeçalho do .fma
COMENTARIO_FMA = f"Matriz agregada da tabela {MTX_SCHEMA}.{MTX_TABELA}"

print(f"Matriz origem: {MTX_SCHEMA}.{MTX_TABELA}")
print(f"Pasta saída:   {OUTPUT_DIR}")
print(f"Arquivos:      {NOME_BASE}.csv  |  {NOME_BASE}.fma")

Matriz origem: VISUM.mtx_od_pequi
Pasta saída:   C:\Users\zeno.filho\Downloads\matrizes_exportadas
Arquivos:      mtx_od_pequi.csv  |  mtx_od_pequi.fma


## 2. Leitura da matriz

In [ ]:
sql = f'''
SELECT {MTX_COL_O} AS o, {MTX_COL_D} AS d, {MTX_COL_VIAGENS} AS viagens
FROM "{MTX_SCHEMA}"."{MTX_TABELA}"
ORDER BY {MTX_COL_O}, {MTX_COL_D};
'''
df = run_sql(sql)
print(f"Linhas carregadas: {len(df):,}")
print(f"Soma total de viagens: {df['viagens'].sum():,.2f}")
df.head()

In [ ]:
csv_path = OUTPUT_DIR / f"{NOME_BASE}.csv"

df.to_csv(
    csv_path,
    sep=";",
    decimal=",",
    index=False,
    encoding="utf-8-sig",   # utf-8 com BOM para o Excel abrir sem problema de acentos
)

print(f"✅ CSV salvo: {csv_path}")
print(f"   Tamanho: {csv_path.stat().st_size / 1024:.1f} KB")

In [ ]:
# Lista ordenada de todas as zonas (união de origens e destinos observados)
zonas = sorted(set(df["o"]).union(set(df["d"])))
n_zonas = len(zonas)
print(f"Número de zonas: {n_zonas}")

if INCLUIR_ZEROS:
    # Expande para todos os pares, preenchendo com 0 onde ausente
    idx = pd.MultiIndex.from_product([zonas, zonas], names=["o", "d"])
    df_full = (
        df.set_index(["o", "d"])
          .reindex(idx, fill_value=0.0)
          .reset_index()
    )
    print(f"Pares expandidos: {len(df_full):,} (N² = {n_zonas * n_zonas:,})")
else:
    df_full = df.copy()
    print(f"Pares não-zero: {len(df_full):,}")

df_full.head()

In [18]:
def exportar_matriz(
    mtx_schema, mtx_tabela,
    output_dir, nome_base,
    mtx_col_o="o", mtx_col_d="d", mtx_col_viagens="viagens",
    incluir_zeros=False,
    casas_decimais=3,
):
    """Exporta uma matriz OD do PostgreSQL para CSV e .fma (formato $O;D3 enxuto).

    Gera o formato validado:
      $O;D3\\n
      * From To Value\\n
      <origem> <destino> <valor>
      ...
    
    Características:
      - Sem BOM, sem Factor, sem Number of zones
      - Separadores CRLF (\\r\\n)
      - "\\n" literal no fim das duas linhas de cabeçalho
      - Dados separados por espaço simples
      - Encoding: ASCII/Latin-1 (sem caracteres especiais no corpo)

    Parâmetros:
      mtx_schema, mtx_tabela  : localização da matriz
      output_dir              : pasta de saída
      nome_base               : nome dos arquivos sem extensão
      mtx_col_o/d/viagens     : nomes das colunas da matriz
      incluir_zeros           : False (padrão) = só pares > 0; True = todos os N²
      casas_decimais          : precisão do valor (também reflete no header $O;D<N>)

    Retorna dict com caminhos e estatísticas.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Ler matriz
    sql = f'''
    SELECT {mtx_col_o} AS o, {mtx_col_d} AS d, {mtx_col_viagens} AS viagens
    FROM "{mtx_schema}"."{mtx_tabela}"
    ORDER BY {mtx_col_o}, {mtx_col_d};
    '''
    df = run_sql(sql)

    # 2. CSV (Excel brasileiro)
    csv_path = output_dir / f"{nome_base}.csv"
    df.to_csv(csv_path, sep=";", decimal=",", index=False, encoding="utf-8-sig")

    # 3. Expandir ou filtrar pares
    zonas = sorted(set(df["o"]).union(set(df["d"])))
    n_zonas = len(zonas)
    if incluir_zeros:
        idx = pd.MultiIndex.from_product([zonas, zonas], names=["o", "d"])
        df_full = df.set_index(["o", "d"]).reindex(idx, fill_value=0.0).reset_index()
    else:
        df_full = df[df["viagens"] > 0].copy()

    # 4. Montar .fma no formato validado
    fma_path = output_dir / f"{nome_base}.fma"

    # Cabeçalho: "\n" literal no meio + CRLF no fim
    # Resultado binário: $O;D3\n\r\n* From To Value\n\r\n
    linhas = [
        f"$O;D{casas_decimais}\\n",
        "* From To Value\\n",
    ]

    # Dados: "o d v" separados por espaço simples
    fmt_valor = f"{{:.{casas_decimais}f}}"
    for row in df_full.itertuples(index=False):
        o = int(row.o)
        d = int(row.d)
        v = fmt_valor.format(row.viagens)
        linhas.append(f"{o} {d} {v}")

    # Junta tudo com CRLF e termina com CRLF
    conteudo = "\r\n".join(linhas) + "\r\n"

    # Escreve em binário para garantir os bytes exatos
    with open(fma_path, "wb") as f:
        f.write(conteudo.encode("latin-1"))

    return {
        "csv":           str(csv_path),
        "fma":           str(fma_path),
        "n_zonas":       n_zonas,
        "n_pares":       len(df_full),
        "total_viagens": float(df["viagens"].sum()),
    }


# Exemplo:
# resultado = exportar_matriz(
#     mtx_schema="VISUM",
#     mtx_tabela="mtx_od_pequivisum",
#     output_dir=r"C:\Users\zeno.filho\Downloads\matrizes_exportadas",
#     nome_base="mtx_od_pequivisum",
# )
# print(resultado)

# Passeio

In [ ]:

 resultado = exportar_matriz(
     mtx_schema="VISUM",
     mtx_tabela="mtx_od_pequivisum",
     output_dir=r"C:\Users\zeno.filho\Downloads\matrizes_exportadas",
     nome_base="mtx_od_pequivisum_passeio",
 )
 print(resultado)

# Comercial Leve

In [ ]:
 resultado = exportar_matriz(
     mtx_schema="VISUM",
     mtx_tabela="",
     output_dir=r"C:\Users\zeno.filho\Downloads\matrizes_exportadas",
     nome_base="mtx_od_pequivisum_cl",
 )
 print(resultado)

# Comercial Pesado

In [ ]:
 resultado = exportar_matriz(
     mtx_schema="VISUM",
     mtx_tabela="",
     output_dir=r"C:\Users\zeno.filho\Downloads\matrizes_exportadas",
     nome_base="mtx_od_pequivisum_cp",
 )
 print(resultado)

## Notas finais

- **Codificação do .fma**: uso `cp1252` (Windows-1252) porque é o padrão que o VISUM espera em arquivos de entrada no ambiente Windows. Se o VISUM do seu projeto estiver configurado pra UTF-8, troque no parâmetro `encoding` da escrita.
- **Nomes das zonas**: estou usando o próprio `id` numérico da zona como "nome". Se você tiver uma coluna de nome (ex.: `nome_zona` em `tbl_shp_pequi`), dá pra fazer um `JOIN` na leitura e substituir os números por strings.
- **Tamanho do arquivo**: para zoneamentos grandes (ex.: 5.000 zonas → 25 milhões de pares), considere `INCLUIR_ZEROS=False` para reduzir. O VISUM interpreta pares ausentes como zero.
- **Validação rápida**: depois de importar no VISUM, confira se o total de viagens da matriz bate com o `total_viagens` que a função retorna — se bater, a exportação foi fiel.